# 1. Flow shop: Johnson's rule for two machines

Lecture: *Week 7 - Flow Shop Scheduling*, 'Johnson's Rule' (PDF pages 13–20).

**The problem.** Every job goes through two machines in the same order, first A, then B, and needs a given time
on each. Find the order of the jobs that finishes all of them as early as possible (the smallest makespan).
Johnson's rule (page 14) gives an optimal order for two machines:

1. List the jobs.
2. Pick the shortest time left in the table (break ties arbitrarily). If it is on machine A, schedule the job as
   early as possible; if it is on machine B, as late as possible.
3. Remove the job from the list, and repeat step 2 until no jobs are left.

In the code the times are a NumPy array `M` with two rows (row 0 = machine A, row 1 = machine B) and one column
per job. Jobs are numbered from 0: job 0 = P, 1 = Q, 2 = R, 3 = S, 4 = T.

**How to run it.** NumPy is already available in the browser, so there is nothing to install: use
*Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Each example is solved twice (recursive and iterative version) and both give the same order:

- `[0, 2, 1]`: P R Q, *Example 2* on the slides (pages 19–20), makespan 18;
- `[2, 3, 4, 1, 0]`: R S T Q P, *Example* (1) on the slides (pages 15–18), makespan 30.

*Original script: `5 Source Files/Python/johnsons_rule.py`. Runs in the browser.*


Load NumPy: the times are stored in a NumPy array, and `np.argmin` finds the smallest one.

In [ ]:
import numpy as np

**Version 1: recursive.** The inner function finds the smallest remaining time with `np.argmin`;
`np.unravel_index` turns its position into (row, column) = (machine, job). The job is crossed out by setting
both of its times to `inf` (infinity), so it is never picked again. If the smallest time is on machine A the job
goes to the end of the `front` list (as early as possible among the jobs still to place); if it is on machine B
it goes to the start of the `end` list (as late as possible). The function calls itself until every time is
`inf`, then returns `front + end`.

In [ ]:
# Do Johnson's rule recursively
def johnsons_rule_rec(M):
    
    def johnsons_rule_rec_inner(M, front, end):
        # if there are no more jobs to do, return the list
        if np.min(M) == np.inf:
            return front + end
        
        # find the smallest remaining time: ind = (row = machine, column = job)
        ind = np.unravel_index(np.argmin(M), M.shape)
        # cross the job out on both machines (inf), so it is never picked again
        M[0, ind[1]] = np.inf
        M[1, ind[1]] = np.inf
        
        if ind[0] == 0:
            # shortest time is on machine A: schedule the job as early as possible
            front.append(int(ind[1]))
            return johnsons_rule_rec_inner(M, front, end)
        else:
            # shortest time is on machine B: schedule the job as late as possible
            end.insert(0, int(ind[1]))
            return johnsons_rule_rec_inner(M, front, end)
    
    # copy the array so we don't modify the original
    M = M.copy()
    return johnsons_rule_rec_inner(M, [], [])

**Version 2: iterative.** The same rule with a loop instead of recursion: one job is scheduled per pass, so
the loop runs once per job (`M.shape[1]` is the number of columns, i.e. of jobs). The last part runs the two
examples from the slides and prints each order twice, once per version.

In [ ]:
# Do Johnson's rule iteratively
def johnsons_rule_iter(M):
    # copy the array so we don't modify the original
    M = M.copy()
    front = []
    end = []
    for _ in range(M.shape[1]):
        # one job is scheduled per pass; this check only skips a pass if every job is already scheduled
        if not np.min(M) == np.inf:
            # find the smallest remaining time: ind = (row = machine, column = job)
            ind = np.unravel_index(np.argmin(M), M.shape)
            # cross the job out on both machines (inf), so it is never picked again
            M[0, ind[1]] = np.inf
            M[1, ind[1]] = np.inf

            if ind[0] == 0:
                # shortest time is on machine A: schedule the job as early as possible
                front.append(int(ind[1]))
            else:
                # shortest time is on machine B: schedule the job as late as possible
                end.insert(0, int(ind[1]))

    return front + end

if __name__ == "__main__":
    # Example 2 on the slides (jobs P, Q, R; PDF pages 19-20): expected [0, 2, 1] = P R Q
    M = np.array([[2, 4, 6],              # times on machine A
        [3, 3, 7]], dtype=float)          # times on machine B
    print(johnsons_rule_rec(M))
    print(johnsons_rule_iter(M))

    # Example 1 on the slides (jobs P to T; PDF pages 15-18): expected [2, 3, 4, 1, 0] = R S T Q P
    M = np.array([[5, 8, 1, 4, 6],        # times on machine A
        [2, 3, 11, 6, 7]], dtype=float)   # times on machine B
    
    print(johnsons_rule_rec(M))
    print(johnsons_rule_iter(M))

## Try it

Use the functions on flow shop tutorial Q1. Add a new cell with

```python
M = np.array([[5, 7, 2, 4, 7],     # machine A, jobs P, Q, R, S, T
              [7, 5, 3, 8, 4]], dtype=float)   # machine B
print(johnsons_rule_iter(M))
```

Which order do you get? Does it match the answer sheet?

<details><summary>Answer</summary>

`[2, 3, 0, 1, 4]`, i.e. R S P Q T, the order on the answer sheet (makespan 30). The table has two ties: 4 (S on
A, T on B) and 5 (P on A, Q on B). `np.argmin` takes the first smallest value row by row, so a tie between the
two machines is always settled in favour of machine A. Here it makes no difference: in each tie one job goes to
the front and the other to the back either way.
</details>
